[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tsilva/aiml-notebooks/blob/main/notebooks/image-reconstruction.ipynb)

# Image Reconstruction with Autoencoders

This notebook explores image reconstruction using three types of autoencoders:
1. **Vanilla Autoencoder** - Basic compression and reconstruction
2. **Variational Autoencoder (VAE)** - Probabilistic latent space with generation capabilities
3. **Vector-Quantized VAE (VQ-VAE)** - Discrete latent space with learned codebook

We'll build understanding progressively, starting from simple concepts and advancing to state-of-the-art architectures.

## Learning Objectives

By the end of this notebook, you will:
- Understand autoencoder architectures and their purpose
- Learn about compression and dimensionality reduction
- Implement three types of autoencoders from scratch
- Explore and visualize latent space representations
- Perform latent space interpolation and manipulation
- Compare different autoencoder approaches
- Apply autoencoders to real-world tasks (denoising, anomaly detection)

## Part 1: Setup and Prerequisites

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
torch.manual_seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])

# Check device availability
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"Using CUDA GPU: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device('mps')
    print("Using MPS (Apple Silicon GPU)")
else:
    device = torch.device('cpu')
    print("Using CPU")

print(f"PyTorch version: {torch.__version__}")

In [ ]:
# Configuration dictionary# All hyperparameters and settings for the notebook are defined hereCONFIG = {    # General settings    'seed': 42,        # Dataset settings    'dataset_name': 'MNIST',    'data_dir': './tmp/data',    'train_split': 0.9,    'val_split': 0.1,        # Model architecture (Autoencoder)    'latent_dim': 32,    'encoder_channels': [32, 64, 128],    'decoder_channels': [128, 64, 32],        # Training settings    'batch_size': 128,    'learning_rate': 0.001,    'num_epochs': 1,        # Visualization    'num_examples': 10,}print("Configuration loaded:")for key, value in CONFIG.items():    print(f"  {key}: {value}")

## Part 2: What Are Autoencoders?

### Theory: Autoencoders Explained

An **autoencoder** is a neural network designed to learn efficient representations of data by compressing it into a lower-dimensional space (encoding) and then reconstructing it back to the original form (decoding).

**Architecture Components:**
1. **Encoder**: Maps input $x$ to latent representation $z$: $z = f_{\text{enc}}(x)$
2. **Latent Space**: Lower-dimensional representation (bottleneck)
3. **Decoder**: Reconstructs from latent: $\hat{x} = f_{\text{dec}}(z)$

**Objective:**
$$\mathcal{L} = \|x - \hat{x}\|^2$$

The network learns to minimize reconstruction error, forcing the bottleneck to capture the most important features.

**Why Use Autoencoders?**
- **Dimensionality Reduction**: Like PCA, but non-linear
- **Feature Learning**: Discover useful representations without labels
- **Denoising**: Learn to remove noise from data
- **Anomaly Detection**: Normal data reconstructs well, anomalies don't
- **Generation**: VAEs can sample new data points
- **Compression**: Reduce storage/transmission costs

### Visualization: Autoencoder Flow

```
Input Image (28×28=784)  →  Encoder  →  Latent (e.g., 32)  →  Decoder  →  Reconstructed (784)
     [High-D]                             [Low-D]                              [High-D]
                                         Bottleneck
                                    (compressed info)
```

The bottleneck forces the network to learn a compressed, meaningful representation.

## Part 3: Load and Prepare MNIST Dataset

In [ ]:
# Transform: Convert images to tensors and normalize to [0, 1]
transform = transforms.Compose([
    transforms.ToTensor(),
])

# Load MNIST dataset
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# Create data loaders
batch_size=CONFIG['batch_size']
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Image shape: {train_dataset[0][0].shape}")  # (1, 28, 28)
print(f"Batch size: {batch_size}")

### Visualize Sample Images

In [ ]:
# Display a grid of sample images
fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i, ax in enumerate(axes.flat):
    img, label = train_dataset[i]
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(f"Label: {label}")
    ax.axis('off')
plt.tight_layout()
plt.show()

print("MNIST contains handwritten digits (0-9)")
print("Each image is 28×28 pixels (grayscale)")

## Part 4: Vanilla Autoencoder

### Theory: Basic Autoencoder Architecture

A vanilla autoencoder consists of:

**Encoder:**
- Flatten input: 28×28 → 784
- Linear layers with decreasing dimensions
- Activation functions (ReLU)
- Output: Latent vector $z \in \mathbb{R}^d$ (e.g., $d=32$)

**Decoder:**
- Start from latent vector
- Linear layers with increasing dimensions
- Final layer: 784 units
- Sigmoid activation (output in [0,1])

**Loss Function:**
$$\mathcal{L}_{\text{MSE}} = \frac{1}{n}\sum_{i=1}^{n}(x_i - \hat{x}_i)^2$$

Or Binary Cross-Entropy (BCE) for pixel values:
$$\mathcal{L}_{\text{BCE}} = -\sum_{i=1}^{n}[x_i \log(\hat{x}_i) + (1-x_i)\log(1-\hat{x}_i)]$$

BCE works well for images normalized to [0,1].

### Implementation: Vanilla Autoencoder

In [ ]:
class VanillaAutoencoder(nn.Module):
    def __init__(self, input_dim=784, latent_dim=CONFIG['latent_dim']):
        super(VanillaAutoencoder, self).__init__()
        
        self.latent_dim = latent_dim
        
        # Encoder: 784 → 512 → 256 → latent_dim
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, latent_dim)
        )
        
        # Decoder: latent_dim → 256 → 512 → 784
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, input_dim),
            nn.Sigmoid()  # Output in [0, 1]
        )
    
    def encode(self, x):
        # Flatten and encode
        x = x.view(x.size(0), -1)  # (batch, 1, 28, 28) → (batch, 784)
        return self.encoder(x)
    
    def decode(self, z):
        # Decode and reshape
        x = self.decoder(z)
        return x.view(x.size(0), 1, 28, 28)  # (batch, 784) → (batch, 1, 28, 28)
    
    def forward(self, x):
        z = self.encode(x)
        x_recon = self.decode(z)
        return x_recon, z

# Create model
vanilla_ae = VanillaAutoencoder(latent_dim=CONFIG['latent_dim']).to(device)
print(vanilla_ae)
print(f"\nTotal parameters: {sum(p.numel() for p in vanilla_ae.parameters()):,}")

### Training Function

In [ ]:
def train_epoch(model, train_loader, optimizer, device):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    
    for batch_idx, (data, _) in enumerate(train_loader):
        data = data.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        recon, _ = model(data)
        
        # Compute loss (BCE for images in [0,1])
        loss = F.binary_cross_entropy(recon, data, reduction='sum') / data.size(0)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(train_loader)

def test_epoch(model, test_loader, device):
    """Evaluate on test set."""
    model.eval()
    total_loss = 0
    
    with torch.no_grad():
        for data, _ in test_loader:
            data = data.to(device)
            recon, _ = model(data)
            loss = F.binary_cross_entropy(recon, data, reduction='sum') / data.size(0)
            total_loss += loss.item()
    
    return total_loss / len(test_loader)

print("Training functions defined.")

### Train Vanilla Autoencoder

In [ ]:
# Training hyperparameters
learning_rate = 1e-3
num_epochs = 20

optimizer = optim.Adam(vanilla_ae.parameters(), lr=learning_rate)

# Training loop
train_losses = []
test_losses = []

for epoch in tqdm(range(num_epochs), desc="Training Vanilla AE"):
    train_loss = train_epoch(vanilla_ae, train_loader, optimizer, device)
    test_loss = test_epoch(vanilla_ae, test_loader, device)
    
    train_losses.append(train_loss)
    test_losses.append(test_loss)
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {train_loss:.4f}, Test Loss: {test_loss:.4f}")

print("\nTraining complete!")

### Visualize Training Progress

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(train_losses, label='Train Loss', linewidth=2)
plt.plot(test_losses, label='Test Loss', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss (BCE)')
plt.title('Vanilla Autoencoder Training')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print(f"Final train loss: {train_losses[-1]:.4f}")
print(f"Final test loss: {test_losses[-1]:.4f}")

## Part 5: Analyzing Reconstruction Quality

Let's visualize how well the autoencoder reconstructs images.

In [ ]:
def visualize_reconstructions(model, dataset, device, n_samples=10):
    """Display original and reconstructed images side by side."""
    model.eval()
    
    # Get random samples
    indices = np.random.choice(len(dataset), n_samples, replace=False)
    images = torch.stack([dataset[i][0] for i in indices]).to(device)
    
    # Reconstruct
    with torch.no_grad():
        recons, _ = model(images)
    
    # Plot
    fig, axes = plt.subplots(2, n_samples, figsize=(n_samples*1.5, 3))
    
    for i in range(n_samples):
        # Original
        axes[0, i].imshow(images[i].cpu().squeeze(), cmap='gray')
        axes[0, i].axis('off')
        if i == 0:
            axes[0, i].set_ylabel('Original', fontsize=12)
        
        # Reconstruction
        axes[1, i].imshow(recons[i].cpu().squeeze(), cmap='gray')
        axes[1, i].axis('off')
        if i == 0:
            axes[1, i].set_ylabel('Reconstructed', fontsize=12)
    
    plt.tight_layout()
    plt.show()

visualize_reconstructions(vanilla_ae, test_dataset, device, n_samples=10)

### Observation

Notice how the reconstructed images capture the overall structure but may lose some fine details. This is expected: the bottleneck (32 dimensions) forces the model to compress 784 dimensions of information.

**Compression Ratio:** 784 / 32 = 24.5× compression!

## Part 6: Latent Space Analysis

### Theory: Understanding the Latent Space

The **latent space** is the compressed representation learned by the encoder. For our autoencoder:
- Input: 784 dimensions (28×28 pixels)
- Latent: 32 dimensions
- Output: 784 dimensions (reconstructed)

The latent space captures the most important features needed to reconstruct the input. We can:
1. **Visualize** it using dimensionality reduction (t-SNE, PCA)
2. **Interpolate** between points to generate new images
3. **Manipulate** it to change image properties

Let's explore these capabilities.

### Extract Latent Representations

In [ ]:
def extract_latent_representations(model, dataset, device, max_samples=5000):
    """Extract latent vectors and labels for a subset of data."""
    model.eval()
    
    latents = []
    labels = []
    
    # Use a subset for faster computation
    subset = Subset(dataset, range(min(max_samples, len(dataset))))
    loader = DataLoader(subset, batch_size=CONFIG['batch_size'], shuffle=False)
    
    with torch.no_grad():
        for data, label in tqdm(loader, desc="Extracting latents"):
            data = data.to(device)
            z = model.encode(data)
            latents.append(z.cpu().numpy())
            labels.append(label.numpy())
    
    latents = np.concatenate(latents, axis=0)
    labels = np.concatenate(labels, axis=0)
    
    return latents, labels

# Extract latent representations
latents, labels = extract_latent_representations(vanilla_ae, test_dataset, device)
print(f"Latent representations shape: {latents.shape}")
print(f"Labels shape: {labels.shape}")

### Visualize Latent Space with t-SNE

In [ ]:
# Apply t-SNE to reduce to 2D
print("Running t-SNE (this may take a minute)...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
latents_2d = tsne.fit_transform(latents)

# Plot
plt.figure(figsize=(12, 10))
scatter = plt.scatter(latents_2d[:, 0], latents_2d[:, 1], c=labels, cmap='tab10', alpha=0.6, s=5)
plt.colorbar(scatter, label='Digit Class')
plt.title('Vanilla Autoencoder Latent Space (t-SNE)', fontsize=14)
plt.xlabel('t-SNE Dimension 1')
plt.ylabel('t-SNE Dimension 2')
plt.grid(alpha=0.3)
plt.show()

print("Each color represents a different digit (0-9).")
print("Notice how similar digits cluster together in latent space.")

### Visualize with PCA

In [ ]:
# Apply PCA to reduce to 2D
pca = PCA(n_components=2)
latents_pca = pca.fit_transform(latents)

# Plot
plt.figure(figsize=(12, 10))
scatter = plt.scatter(latents_pca[:, 0], latents_pca[:, 1], c=labels, cmap='tab10', alpha=0.6, s=5)
plt.colorbar(scatter, label='Digit Class')
plt.title('Vanilla Autoencoder Latent Space (PCA)', fontsize=14)
plt.xlabel('PC1 ({:.1f}% var)'.format(pca.explained_variance_ratio_[0]*100))
plt.ylabel('PC2 ({:.1f}% var)'.format(pca.explained_variance_ratio_[1]*100))
plt.grid(alpha=0.3)
plt.show()

print(f"Total variance explained by 2 PCs: {pca.explained_variance_ratio_.sum()*100:.1f}%")

## Part 7: Latent Space Interpolation

### Theory: Interpolation in Latent Space

Since the decoder maps latent vectors to images, we can:
1. Take two images
2. Encode them to latent vectors $z_1$ and $z_2$
3. Create intermediate vectors: $z_t = (1-t)z_1 + tz_2$ for $t \in [0,1]$
4. Decode each $z_t$ to get interpolated images

This shows how the latent space represents a smooth transition between different images.

In [ ]:
def interpolate_latents(model, img1, img2, device, n_steps=10):
    """Interpolate between two images in latent space."""
    model.eval()
    
    # Encode both images
    with torch.no_grad():
        img1 = img1.unsqueeze(0).to(device)
        img2 = img2.unsqueeze(0).to(device)
        z1 = model.encode(img1)
        z2 = model.encode(img2)
        
        # Create interpolation steps
        interpolations = []
        for t in np.linspace(0, 1, n_steps):
            z_t = (1 - t) * z1 + t * z2
            img_t = model.decode(z_t)
            interpolations.append(img_t.cpu().squeeze())
    
    return interpolations

# Select two different digits
idx1, idx2 = 0, 5
img1, label1 = test_dataset[idx1]
img2, label2 = test_dataset[idx2]

# Interpolate
interpolations = interpolate_latents(vanilla_ae, img1, img2, device, n_steps=10)

# Visualize
fig, axes = plt.subplots(1, len(interpolations), figsize=(15, 2))
for i, ax in enumerate(axes):
    ax.imshow(interpolations[i], cmap='gray')
    ax.axis('off')
    if i == 0:
        ax.set_title(f'Digit {label1}', fontsize=10)
    elif i == len(interpolations) - 1:
        ax.set_title(f'Digit {label2}', fontsize=10)
plt.suptitle('Latent Space Interpolation', fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

print(f"Smoothly transitioning from '{label1}' to '{label2}' in latent space.")

### Reflection Question

**Q:** What do you observe in the interpolated images? Are they realistic?

**A:** The interpolations show a gradual morph between digits. However, intermediate images may look unrealistic or blurry. This is a limitation of vanilla autoencoders—the latent space is not structured to produce realistic samples at arbitrary points. **Variational Autoencoders (VAEs)** address this by imposing structure on the latent space.

## Part 8: Variational Autoencoder (VAE)

### Theory: From Deterministic to Probabilistic

**Problem with Vanilla AE:**
- Encoder outputs a single point $z$ for each input
- Latent space may have "holes" (regions that don't decode to realistic images)
- Cannot sample new images easily

**VAE Solution:**
- Encoder outputs a **distribution** $q(z|x) = \mathcal{N}(\mu, \sigma^2)$
- Sample $z \sim q(z|x)$ during training
- Decoder learns $p(x|z)$

**Objective (ELBO):**
$$\mathcal{L}_{\text{VAE}} = \underbrace{\mathbb{E}_{q(z|x)}[\log p(x|z)]}_\text{Reconstruction Loss} - \underbrace{D_{KL}(q(z|x) \| p(z))}_\text{KL Divergence}$$

Where:
- **Reconstruction Loss**: How well we reconstruct input (like vanilla AE)
- **KL Divergence**: Regularization forcing $q(z|x)$ to be close to prior $p(z) = \mathcal{N}(0, I)$

**Reparameterization Trick:**
To allow gradients to flow through sampling:
$$z = \mu + \sigma \odot \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$$

This makes the sampling operation differentiable.

### KL Divergence Explained

For Gaussian distributions:
$$D_{KL}(\mathcal{N}(\mu, \sigma^2) \| \mathcal{N}(0, 1)) = \frac{1}{2}\sum_{i=1}^{d}(\mu_i^2 + \sigma_i^2 - \log(\sigma_i^2) - 1)$$

**Intuition:**
- Pulls $\mu$ towards 0
- Pulls $\sigma$ towards 1
- Creates a smooth, continuous latent space
- Enables sampling: $z \sim \mathcal{N}(0, I)$ should decode to realistic images

### Implementation: VAE

In [ ]:
class VAE(nn.Module):
    def __init__(self, input_dim=784, latent_dim=CONFIG['latent_dim']):
        super(VAE, self).__init__()
        
        self.latent_dim = latent_dim
        
        # Encoder: outputs mean and log-variance
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
        )
        
        self.fc_mu = nn.Linear(256, latent_dim)
        self.fc_logvar = nn.Linear(256, latent_dim)
        
        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, input_dim),
            nn.Sigmoid()
        )
    
    def encode(self, x):
        """Encode to mean and log-variance."""
        x = x.view(x.size(0), -1)
        h = self.encoder(x)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar
    
    def reparameterize(self, mu, logvar):
        """Reparameterization trick: z = mu + sigma * epsilon."""
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def decode(self, z):
        """Decode latent to reconstruction."""
        x = self.decoder(z)
        return x.view(x.size(0), 1, 28, 28)
    
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_recon = self.decode(z)
        return x_recon, mu, logvar

# Create VAE
vae = VAE(latent_dim=CONFIG['latent_dim']).to(device)
print(vae)
print(f"\nTotal parameters: {sum(p.numel() for p in vae.parameters()):,}")

### VAE Loss Function

In [ ]:
def vae_loss(recon_x, x, mu, logvar):
    """
    Compute VAE loss = Reconstruction Loss + KL Divergence.
    
    Args:
        recon_x: Reconstructed images
        x: Original images
        mu: Mean of latent distribution
        logvar: Log-variance of latent distribution
    
    Returns:
        total_loss, recon_loss, kl_loss
    """
    # Reconstruction loss (BCE)
    recon_loss = F.binary_cross_entropy(recon_x, x, reduction='sum') / x.size(0)
    
    # KL divergence: -0.5 * sum(1 + log(sigma^2) - mu^2 - sigma^2)
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x.size(0)
    
    return recon_loss + kl_loss, recon_loss, kl_loss

print("VAE loss function defined.")

### Training Functions for VAE

In [ ]:
def train_epoch_vae(model, train_loader, optimizer, device):
    """Train VAE for one epoch."""
    model.train()
    total_loss = 0
    total_recon = 0
    total_kl = 0
    
    for batch_idx, (data, _) in enumerate(train_loader):
        data = data.to(device)
        
        optimizer.zero_grad()
        recon, mu, logvar = model(data)
        
        loss, recon_loss, kl_loss = vae_loss(recon, data, mu, logvar)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        total_recon += recon_loss.item()
        total_kl += kl_loss.item()
    
    n = len(train_loader)
    return total_loss/n, total_recon/n, total_kl/n

def test_epoch_vae(model, test_loader, device):
    """Evaluate VAE on test set."""
    model.eval()
    total_loss = 0
    total_recon = 0
    total_kl = 0
    
    with torch.no_grad():
        for data, _ in test_loader:
            data = data.to(device)
            recon, mu, logvar = model(data)
            loss, recon_loss, kl_loss = vae_loss(recon, data, mu, logvar)
            
            total_loss += loss.item()
            total_recon += recon_loss.item()
            total_kl += kl_loss.item()
    
    n = len(test_loader)
    return total_loss/n, total_recon/n, total_kl/n

print("VAE training functions defined.")

### Train VAE

In [ ]:
# Training hyperparameters
learning_rate = 1e-3
num_epochs = 20

optimizer = optim.Adam(vae.parameters(), lr=learning_rate)

# Training loop
train_losses_vae = []
test_losses_vae = []
train_recon_losses = []
train_kl_losses = []

for epoch in tqdm(range(num_epochs), desc="Training VAE"):
    train_loss, train_recon, train_kl = train_epoch_vae(vae, train_loader, optimizer, device)
    test_loss, test_recon, test_kl = test_epoch_vae(vae, test_loader, device)
    
    train_losses_vae.append(train_loss)
    test_losses_vae.append(test_loss)
    train_recon_losses.append(train_recon)
    train_kl_losses.append(train_kl)
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {train_loss:.4f} (Recon: {train_recon:.4f}, KL: {train_kl:.4f}), Test Loss: {test_loss:.4f}")

print("\nVAE training complete!")

### Visualize VAE Training

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Total loss
axes[0].plot(train_losses_vae, label='Train Loss', linewidth=2)
axes[0].plot(test_losses_vae, label='Test Loss', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('VAE Total Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Loss components
axes[1].plot(train_recon_losses, label='Reconstruction Loss', linewidth=2)
axes[1].plot(train_kl_losses, label='KL Divergence', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('VAE Loss Components (Train)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Final train loss: {train_losses_vae[-1]:.4f}")
print(f"Final test loss: {test_losses_vae[-1]:.4f}")

### Visualize VAE Reconstructions

In [ ]:
def visualize_reconstructions_vae(model, dataset, device, n_samples=10):
    """Display original and VAE reconstructed images."""
    model.eval()
    
    indices = np.random.choice(len(dataset), n_samples, replace=False)
    images = torch.stack([dataset[i][0] for i in indices]).to(device)
    
    with torch.no_grad():
        recons, _, _ = model(images)
    
    fig, axes = plt.subplots(2, n_samples, figsize=(n_samples*1.5, 3))
    
    for i in range(n_samples):
        axes[0, i].imshow(images[i].cpu().squeeze(), cmap='gray')
        axes[0, i].axis('off')
        if i == 0:
            axes[0, i].set_ylabel('Original', fontsize=12)
        
        axes[1, i].imshow(recons[i].cpu().squeeze(), cmap='gray')
        axes[1, i].axis('off')
        if i == 0:
            axes[1, i].set_ylabel('VAE Recon', fontsize=12)
    
    plt.tight_layout()
    plt.show()

visualize_reconstructions_vae(vae, test_dataset, device, n_samples=10)

## Part 9: VAE Latent Space Analysis

The key advantage of VAE is its **structured latent space**. Let's visualize it.

In [ ]:
def extract_latent_representations_vae(model, dataset, device, max_samples=5000):
    """Extract VAE latent representations (using mean, not sampled)."""
    model.eval()
    
    latents = []
    labels = []
    
    subset = Subset(dataset, range(min(max_samples, len(dataset))))
    loader = DataLoader(subset, batch_size=CONFIG['batch_size'], shuffle=False)
    
    with torch.no_grad():
        for data, label in tqdm(loader, desc="Extracting VAE latents"):
            data = data.to(device)
            mu, _ = model.encode(data)
            latents.append(mu.cpu().numpy())
            labels.append(label.numpy())
    
    latents = np.concatenate(latents, axis=0)
    labels = np.concatenate(labels, axis=0)
    
    return latents, labels

# Extract VAE latent representations
latents_vae, labels_vae = extract_latent_representations_vae(vae, test_dataset, device)
print(f"VAE latent representations shape: {latents_vae.shape}")

### Visualize VAE Latent Space with t-SNE

In [ ]:
print("Running t-SNE on VAE latents...")
tsne_vae = TSNE(n_components=2, random_state=42, perplexity=30)
latents_vae_2d = tsne_vae.fit_transform(latents_vae)

plt.figure(figsize=(12, 10))
scatter = plt.scatter(latents_vae_2d[:, 0], latents_vae_2d[:, 1], c=labels_vae, cmap='tab10', alpha=0.6, s=5)
plt.colorbar(scatter, label='Digit Class')
plt.title('VAE Latent Space (t-SNE)', fontsize=14)
plt.xlabel('t-SNE Dimension 1')
plt.ylabel('t-SNE Dimension 2')
plt.grid(alpha=0.3)
plt.show()

print("VAE latent space shows cleaner clustering due to KL regularization.")

### Compare Vanilla AE vs VAE Latent Spaces

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Vanilla AE
axes[0].scatter(latents_2d[:, 0], latents_2d[:, 1], c=labels, cmap='tab10', alpha=0.6, s=5)
axes[0].set_title('Vanilla Autoencoder Latent Space (t-SNE)', fontsize=12)
axes[0].set_xlabel('t-SNE Dimension 1')
axes[0].set_ylabel('t-SNE Dimension 2')
axes[0].grid(alpha=0.3)

# VAE
scatter = axes[1].scatter(latents_vae_2d[:, 0], latents_vae_2d[:, 1], c=labels_vae, cmap='tab10', alpha=0.6, s=5)
axes[1].set_title('VAE Latent Space (t-SNE)', fontsize=12)
axes[1].set_xlabel('t-SNE Dimension 1')
axes[1].set_ylabel('t-SNE Dimension 2')
axes[1].grid(alpha=0.3)

plt.colorbar(scatter, ax=axes, label='Digit Class')
plt.tight_layout()
plt.show()

print("Notice: VAE latent space tends to have better-separated clusters.")
print("This is due to the KL divergence term regularizing the latent space.")

## Part 10: Sampling from VAE

### Theory: Generative Capability

Unlike vanilla autoencoders, VAEs can **generate** new images:
1. Sample $z \sim \mathcal{N}(0, I)$ from the prior
2. Decode: $\hat{x} = f_{\text{dec}}(z)$

This works because the KL term forces the latent space to match $\mathcal{N}(0, I)$, so random samples should decode to realistic images.

In [ ]:
def sample_vae(model, device, n_samples=16):
    """Generate new images by sampling from the prior."""
    model.eval()
    
    with torch.no_grad():
        # Sample from standard normal
        z = torch.randn(n_samples, model.latent_dim).to(device)
        # Decode
        samples = model.decode(z)
    
    return samples.cpu()

# Generate samples
samples = sample_vae(vae, device, n_samples=16)

# Visualize
fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i, ax in enumerate(axes.flat):
    ax.imshow(samples[i].squeeze(), cmap='gray')
    ax.axis('off')
plt.suptitle('Random Samples from VAE (sampled from N(0,I))', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("These images were generated from random latent vectors, not from the dataset!")

### VAE Latent Space Interpolation

In [ ]:
def interpolate_latents_vae(model, img1, img2, device, n_steps=10):
    """Interpolate between two images in VAE latent space."""
    model.eval()
    
    with torch.no_grad():
        img1 = img1.unsqueeze(0).to(device)
        img2 = img2.unsqueeze(0).to(device)
        
        # Encode to mean (not sampled)
        mu1, _ = model.encode(img1)
        mu2, _ = model.encode(img2)
        
        # Interpolate
        interpolations = []
        for t in np.linspace(0, 1, n_steps):
            z_t = (1 - t) * mu1 + t * mu2
            img_t = model.decode(z_t)
            interpolations.append(img_t.cpu().squeeze())
    
    return interpolations

# Select two images
idx1, idx2 = 0, 5
img1, label1 = test_dataset[idx1]
img2, label2 = test_dataset[idx2]

# Interpolate
interpolations_vae = interpolate_latents_vae(vae, img1, img2, device, n_steps=10)

# Visualize
fig, axes = plt.subplots(1, len(interpolations_vae), figsize=(15, 2))
for i, ax in enumerate(axes):
    ax.imshow(interpolations_vae[i], cmap='gray')
    ax.axis('off')
    if i == 0:
        ax.set_title(f'Digit {label1}', fontsize=10)
    elif i == len(interpolations_vae) - 1:
        ax.set_title(f'Digit {label2}', fontsize=10)
plt.suptitle('VAE Latent Space Interpolation', fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

print("VAE interpolations tend to be smoother and more realistic than vanilla AE.")

## Part 11: Vector-Quantized VAE (VQ-VAE)

### Theory: Discrete Latent Space

**Motivation:**
- VAE uses continuous latent space: $z \in \mathbb{R}^d$
- What if we use **discrete** latent space instead?
- Like a "codebook" of learned vectors

**VQ-VAE Architecture:**
1. **Encoder**: Maps input to continuous vector $z_e$
2. **Vector Quantization**: Find nearest codebook vector $z_q$
3. **Decoder**: Reconstruct from $z_q$

**Codebook:**
- $K$ learned vectors: $e_1, e_2, ..., e_K \in \mathbb{R}^d$
- For encoded $z_e$, find nearest: $z_q = e_k$ where $k = \arg\min_j \|z_e - e_j\|_2$
- Pass $z_q$ to decoder

**Loss Function:**
$$\mathcal{L} = \underbrace{\|x - \hat{x}\|^2}_\text{Reconstruction} + \underbrace{\|sg[z_e] - e\|_2^2}_\text{Codebook Loss} + \beta \underbrace{\|z_e - sg[e]\|_2^2}_\text{Commitment Loss}$$

Where $sg[\cdot]$ is stop-gradient (no backprop).

**Straight-Through Estimator:**
- Quantization is non-differentiable
- Solution: Copy gradients from decoder to encoder: $\frac{\partial \mathcal{L}}{\partial z_e} = \frac{\partial \mathcal{L}}{\partial z_q}$

### VQ-VAE Benefits

- **Discrete representations**: Useful for autoregressive models (like PixelCNN)
- **Avoids posterior collapse**: Common VAE problem
- **Better reconstructions**: Especially for complex data
- **Applications**: Image generation (DALL-E), speech synthesis

### Implementation: Vector Quantizer

In [ ]:
class VectorQuantizer(nn.Module):
    """Vector Quantization layer."""
    
    def __init__(self, num_embeddings, embedding_dim, commitment_cost=0.25):
        super(VectorQuantizer, self).__init__()
        
        self.num_embeddings = num_embeddings
        self.embedding_dim = embedding_dim
        self.commitment_cost = commitment_cost
        
        # Codebook (initialized uniformly)
        self.embedding = nn.Embedding(num_embeddings, embedding_dim)
        self.embedding.weight.data.uniform_(-1/num_embeddings, 1/num_embeddings)
    
    def forward(self, z):
        """
        Args:
            z: Encoder output (batch, embedding_dim)
        
        Returns:
            z_q: Quantized vectors
            loss: VQ loss (codebook + commitment)
            perplexity: Measure of codebook usage
        """
        # Flatten input
        z_flattened = z.view(-1, self.embedding_dim)
        
        # Calculate distances to codebook vectors
        # (batch, num_embeddings)
        distances = (torch.sum(z_flattened**2, dim=1, keepdim=True)
                    + torch.sum(self.embedding.weight**2, dim=1)
                    - 2 * torch.matmul(z_flattened, self.embedding.weight.t()))
        
        # Find nearest codebook vectors
        encoding_indices = torch.argmin(distances, dim=1).unsqueeze(1)
        
        # Convert to one-hot
        encodings = torch.zeros(encoding_indices.shape[0], self.num_embeddings, device=z.device)
        encodings.scatter_(1, encoding_indices, 1)
        
        # Quantize
        z_q = torch.matmul(encodings, self.embedding.weight).view(z.shape)
        
        # Loss
        codebook_loss = F.mse_loss(z_q.detach(), z)
        commitment_loss = F.mse_loss(z_q, z.detach())
        loss = codebook_loss + self.commitment_cost * commitment_loss
        
        # Straight-through estimator
        z_q = z + (z_q - z).detach()
        
        # Perplexity (measure of codebook usage)
        avg_probs = torch.mean(encodings, dim=0)
        perplexity = torch.exp(-torch.sum(avg_probs * torch.log(avg_probs + 1e-10)))
        
        return z_q, loss, perplexity

print("Vector Quantizer implemented.")

### Implementation: VQ-VAE Model

In [ ]:
class VQVAE(nn.Module):
    def __init__(self, input_dim=784, latent_dim=CONFIG['latent_dim'], num_embeddings=512, commitment_cost=0.25):
        super(VQVAE, self).__init__()
        
        self.latent_dim = latent_dim
        self.num_embeddings = num_embeddings
        
        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, latent_dim)
        )
        
        # Vector Quantization
        self.vq = VectorQuantizer(num_embeddings, latent_dim, commitment_cost)
        
        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, input_dim),
            nn.Sigmoid()
        )
    
    def encode(self, x):
        x = x.view(x.size(0), -1)
        return self.encoder(x)
    
    def decode(self, z):
        x = self.decoder(z)
        return x.view(x.size(0), 1, 28, 28)
    
    def forward(self, x):
        z_e = self.encode(x)
        z_q, vq_loss, perplexity = self.vq(z_e)
        x_recon = self.decode(z_q)
        return x_recon, vq_loss, perplexity

# Create VQ-VAE
vqvae = VQVAE(latent_dim=CONFIG['latent_dim'], num_embeddings=512).to(device)
print(vqvae)
print(f"\nTotal parameters: {sum(p.numel() for p in vqvae.parameters()):,}")
print(f"Codebook size: {vqvae.num_embeddings} vectors of dimension {vqvae.latent_dim}")

### Training Functions for VQ-VAE

In [ ]:
def train_epoch_vqvae(model, train_loader, optimizer, device):
    """Train VQ-VAE for one epoch."""
    model.train()
    total_loss = 0
    total_recon = 0
    total_vq = 0
    total_perplexity = 0
    
    for batch_idx, (data, _) in enumerate(train_loader):
        data = data.to(device)
        
        optimizer.zero_grad()
        recon, vq_loss, perplexity = model(data)
        
        # Total loss = reconstruction + VQ loss
        recon_loss = F.binary_cross_entropy(recon, data, reduction='sum') / data.size(0)
        loss = recon_loss + vq_loss
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        total_recon += recon_loss.item()
        total_vq += vq_loss.item()
        total_perplexity += perplexity.item()
    
    n = len(train_loader)
    return total_loss/n, total_recon/n, total_vq/n, total_perplexity/n

def test_epoch_vqvae(model, test_loader, device):
    """Evaluate VQ-VAE on test set."""
    model.eval()
    total_loss = 0
    total_recon = 0
    total_vq = 0
    total_perplexity = 0
    
    with torch.no_grad():
        for data, _ in test_loader:
            data = data.to(device)
            recon, vq_loss, perplexity = model(data)
            
            recon_loss = F.binary_cross_entropy(recon, data, reduction='sum') / data.size(0)
            loss = recon_loss + vq_loss
            
            total_loss += loss.item()
            total_recon += recon_loss.item()
            total_vq += vq_loss.item()
            total_perplexity += perplexity.item()
    
    n = len(test_loader)
    return total_loss/n, total_recon/n, total_vq/n, total_perplexity/n

print("VQ-VAE training functions defined.")

### Train VQ-VAE

In [ ]:
# Training hyperparameters
learning_rate = 1e-3
num_epochs = 20

optimizer = optim.Adam(vqvae.parameters(), lr=learning_rate)

# Training loop
train_losses_vqvae = []
test_losses_vqvae = []
train_recon_losses_vq = []
train_vq_losses = []
perplexities = []

for epoch in tqdm(range(num_epochs), desc="Training VQ-VAE"):
    train_loss, train_recon, train_vq, train_perplexity = train_epoch_vqvae(vqvae, train_loader, optimizer, device)
    test_loss, test_recon, test_vq, test_perplexity = test_epoch_vqvae(vqvae, test_loader, device)
    
    train_losses_vqvae.append(train_loss)
    test_losses_vqvae.append(test_loss)
    train_recon_losses_vq.append(train_recon)
    train_vq_losses.append(train_vq)
    perplexities.append(train_perplexity)
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {train_loss:.4f} (Recon: {train_recon:.4f}, VQ: {train_vq:.4f}), Perplexity: {train_perplexity:.2f}")

print("\nVQ-VAE training complete!")

### Visualize VQ-VAE Training

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Total loss
axes[0].plot(train_losses_vqvae, label='Train Loss', linewidth=2)
axes[0].plot(test_losses_vqvae, label='Test Loss', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('VQ-VAE Total Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Loss components
axes[1].plot(train_recon_losses_vq, label='Reconstruction Loss', linewidth=2)
axes[1].plot(train_vq_losses, label='VQ Loss', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('VQ-VAE Loss Components')
axes[1].legend()
axes[1].grid(alpha=0.3)

# Perplexity
axes[2].plot(perplexities, linewidth=2, color='green')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Perplexity')
axes[2].set_title('Codebook Perplexity')
axes[2].grid(alpha=0.3)
axes[2].axhline(y=vqvae.num_embeddings, color='r', linestyle='--', label=f'Max ({vqvae.num_embeddings})')
axes[2].legend()

plt.tight_layout()
plt.show()

print(f"Final train loss: {train_losses_vqvae[-1]:.4f}")
print(f"Final perplexity: {perplexities[-1]:.2f} / {vqvae.num_embeddings}")
print(f"Codebook usage: {perplexities[-1]/vqvae.num_embeddings*100:.1f}%")

### Understanding Perplexity

**Perplexity** measures how many codebook vectors are actively used:
- Max: All `num_embeddings` vectors used equally
- Low: Only a few vectors dominate (codebook collapse)
- Ideal: High perplexity indicates diverse codebook usage

A healthy VQ-VAE should maintain reasonably high perplexity throughout training.

### Visualize VQ-VAE Reconstructions

In [ ]:
def visualize_reconstructions_vqvae(model, dataset, device, n_samples=10):
    """Display original and VQ-VAE reconstructed images."""
    model.eval()
    
    indices = np.random.choice(len(dataset), n_samples, replace=False)
    images = torch.stack([dataset[i][0] for i in indices]).to(device)
    
    with torch.no_grad():
        recons, _, _ = model(images)
    
    fig, axes = plt.subplots(2, n_samples, figsize=(n_samples*1.5, 3))
    
    for i in range(n_samples):
        axes[0, i].imshow(images[i].cpu().squeeze(), cmap='gray')
        axes[0, i].axis('off')
        if i == 0:
            axes[0, i].set_ylabel('Original', fontsize=12)
        
        axes[1, i].imshow(recons[i].cpu().squeeze(), cmap='gray')
        axes[1, i].axis('off')
        if i == 0:
            axes[1, i].set_ylabel('VQ-VAE Recon', fontsize=12)
    
    plt.tight_layout()
    plt.show()

visualize_reconstructions_vqvae(vqvae, test_dataset, device, n_samples=10)

## Part 12: Analyzing VQ-VAE Codebook Usage

In [ ]:
def analyze_codebook_usage(model, dataset, device, max_samples=5000):
    """Analyze which codebook vectors are used."""
    model.eval()
    
    # Count codebook vector usage
    codebook_counts = torch.zeros(model.num_embeddings)
    
    subset = Subset(dataset, range(min(max_samples, len(dataset))))
    loader = DataLoader(subset, batch_size=CONFIG['batch_size'], shuffle=False)
    
    with torch.no_grad():
        for data, _ in tqdm(loader, desc="Analyzing codebook"):
            data = data.to(device)
            
            # Encode
            z_e = model.encode(data)
            z_flattened = z_e.view(-1, model.latent_dim)
            
            # Find nearest codebook vectors
            distances = (torch.sum(z_flattened**2, dim=1, keepdim=True)
                        + torch.sum(model.vq.embedding.weight**2, dim=1)
                        - 2 * torch.matmul(z_flattened, model.vq.embedding.weight.t()))
            
            encoding_indices = torch.argmin(distances, dim=1)
            
            # Count
            for idx in encoding_indices:
                codebook_counts[idx.item()] += 1
    
    return codebook_counts.numpy()

# Analyze codebook
codebook_usage = analyze_codebook_usage(vqvae, test_dataset, device)

# Visualize
plt.figure(figsize=(14, 4))
plt.bar(range(len(codebook_usage)), codebook_usage, width=1.0)
plt.xlabel('Codebook Index')
plt.ylabel('Usage Count')
plt.title('VQ-VAE Codebook Vector Usage')
plt.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

# Statistics
used_vectors = np.sum(codebook_usage > 0)
print(f"Total codebook vectors: {vqvae.num_embeddings}")
print(f"Used vectors: {used_vectors} ({used_vectors/vqvae.num_embeddings*100:.1f}%)")
print(f"Unused vectors: {vqvae.num_embeddings - used_vectors}")
print(f"Most used vector: {np.max(codebook_usage):.0f} times")
print(f"Average usage: {np.mean(codebook_usage[codebook_usage > 0]):.1f} times")

### Visualize Most Common Codebook Vectors

In [ ]:
# Get top 16 most-used codebook vectors
top_indices = np.argsort(codebook_usage)[-16:][::-1]

# Decode them to images
vqvae.eval()
with torch.no_grad():
    # Get codebook vectors
    codebook_vectors = vqvae.vq.embedding.weight[top_indices].to(device)
    # Decode
    images = vqvae.decode(codebook_vectors)

# Visualize
fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i].cpu().squeeze(), cmap='gray')
    ax.set_title(f'#{top_indices[i]}\n({codebook_usage[top_indices[i]]:.0f})', fontsize=8)
    ax.axis('off')
plt.suptitle('Most Frequently Used Codebook Vectors (decoded to images)', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print("These are the 16 most-used discrete latent representations learned by VQ-VAE.")

## Part 13: Comparing All Three Autoencoders

Let's compare the three approaches side-by-side.

### Reconstruction Quality Comparison

In [ ]:
# Select random test images
n_samples = 8
indices = np.random.choice(len(test_dataset), n_samples, replace=False)
images = torch.stack([test_dataset[i][0] for i in indices]).to(device)

# Get reconstructions from all three models
vanilla_ae.eval()
vae.eval()
vqvae.eval()

with torch.no_grad():
    recon_vanilla, _ = vanilla_ae(images)
    recon_vae, _, _ = vae(images)
    recon_vqvae, _, _ = vqvae(images)

# Visualize
fig, axes = plt.subplots(4, n_samples, figsize=(n_samples*1.5, 6))

for i in range(n_samples):
    # Original
    axes[0, i].imshow(images[i].cpu().squeeze(), cmap='gray')
    axes[0, i].axis('off')
    if i == 0:
        axes[0, i].set_ylabel('Original', fontsize=12)
    
    # Vanilla AE
    axes[1, i].imshow(recon_vanilla[i].cpu().squeeze(), cmap='gray')
    axes[1, i].axis('off')
    if i == 0:
        axes[1, i].set_ylabel('Vanilla AE', fontsize=12)
    
    # VAE
    axes[2, i].imshow(recon_vae[i].cpu().squeeze(), cmap='gray')
    axes[2, i].axis('off')
    if i == 0:
        axes[2, i].set_ylabel('VAE', fontsize=12)
    
    # VQ-VAE
    axes[3, i].imshow(recon_vqvae[i].cpu().squeeze(), cmap='gray')
    axes[3, i].axis('off')
    if i == 0:
        axes[3, i].set_ylabel('VQ-VAE', fontsize=12)

plt.suptitle('Reconstruction Comparison: Vanilla AE vs VAE vs VQ-VAE', fontsize=14, y=1.00)
plt.tight_layout()
plt.show()

### Quantitative Comparison

In [ ]:
# Compute test losses for all models
def compute_test_loss(model, test_loader, device, model_type='vanilla'):
    """Compute average test loss."""
    model.eval()
    total_loss = 0
    
    with torch.no_grad():
        for data, _ in test_loader:
            data = data.to(device)
            
            if model_type == 'vanilla':
                recon, _ = model(data)
                loss = F.binary_cross_entropy(recon, data, reduction='sum') / data.size(0)
            elif model_type == 'vae':
                recon, mu, logvar = model(data)
                loss, _, _ = vae_loss(recon, data, mu, logvar)
            elif model_type == 'vqvae':
                recon, vq_loss, _ = model(data)
                recon_loss = F.binary_cross_entropy(recon, data, reduction='sum') / data.size(0)
                loss = recon_loss + vq_loss
            
            total_loss += loss.item()
    
    return total_loss / len(test_loader)

# Compute losses
vanilla_test_loss = compute_test_loss(vanilla_ae, test_loader, device, 'vanilla')
vae_test_loss = compute_test_loss(vae, test_loader, device, 'vae')
vqvae_test_loss = compute_test_loss(vqvae, test_loader, device, 'vqvae')

# Create comparison table
import pandas as pd

comparison_df = pd.DataFrame({
    'Model': ['Vanilla AE', 'VAE', 'VQ-VAE'],
    'Test Loss': [vanilla_test_loss, vae_test_loss, vqvae_test_loss],
    'Latent Type': ['Continuous', 'Probabilistic', 'Discrete'],
    'Generation': ['Poor', 'Good', 'Good (with prior)'],
    'Parameters': [
        sum(p.numel() for p in vanilla_ae.parameters()),
        sum(p.numel() for p in vae.parameters()),
        sum(p.numel() for p in vqvae.parameters())
    ]
})

print("\n" + "="*70)
print("AUTOENCODER COMPARISON")
print("="*70)
print(comparison_df.to_string(index=False))
print("="*70)

### Key Differences Summary

| Feature | Vanilla AE | VAE | VQ-VAE |
|---------|------------|-----|--------|
| **Latent Space** | Continuous (deterministic) | Continuous (probabilistic) | Discrete (codebook) |
| **Loss** | Reconstruction only | Reconstruction + KL | Reconstruction + VQ |
| **Generation** | Poor (unstructured latent) | Good (sample from N(0,I)) | Good (with learned prior) |
| **Interpolation** | Possible but may be unrealistic | Smooth and realistic | Discrete jumps |
| **Best For** | Compression, denoising | Generation, interpolation | Discrete representations, generation |
| **Complexity** | Simple | Moderate | Higher |

## Part 14: Applications

### Application 1: Image Denoising

Autoencoders can learn to remove noise from images.

In [ ]:
def add_noise(images, noise_factor=0.3):
    """Add Gaussian noise to images."""
    noisy = images + noise_factor * torch.randn_like(images)
    return torch.clamp(noisy, 0., 1.)

# Get clean images
n_samples = 8
indices = np.random.choice(len(test_dataset), n_samples, replace=False)
clean_images = torch.stack([test_dataset[i][0] for i in indices]).to(device)

# Add noise
noisy_images = add_noise(clean_images, noise_factor=0.5)

# Denoise with VAE
vae.eval()
with torch.no_grad():
    denoised, _, _ = vae(noisy_images)

# Visualize
fig, axes = plt.subplots(3, n_samples, figsize=(n_samples*1.5, 5))

for i in range(n_samples):
    # Clean
    axes[0, i].imshow(clean_images[i].cpu().squeeze(), cmap='gray')
    axes[0, i].axis('off')
    if i == 0:
        axes[0, i].set_ylabel('Clean', fontsize=12)
    
    # Noisy
    axes[1, i].imshow(noisy_images[i].cpu().squeeze(), cmap='gray')
    axes[1, i].axis('off')
    if i == 0:
        axes[1, i].set_ylabel('Noisy', fontsize=12)
    
    # Denoised
    axes[2, i].imshow(denoised[i].cpu().squeeze(), cmap='gray')
    axes[2, i].axis('off')
    if i == 0:
        axes[2, i].set_ylabel('Denoised (VAE)', fontsize=12)

plt.suptitle('Application: Image Denoising with VAE', fontsize=14, y=0.98)
plt.tight_layout()
plt.show()

print("VAE successfully removes noise by projecting to learned latent manifold.")

### Application 2: Anomaly Detection

**Idea:** Normal samples reconstruct well, anomalies don't.

Let's train on only digits 0-8 and test on digit 9 (anomaly).

In [ ]:
# Separate normal (0-8) and anomaly (9) samples
normal_indices = [i for i in range(len(test_dataset)) if test_dataset[i][1] != 9]
anomaly_indices = [i for i in range(len(test_dataset)) if test_dataset[i][1] == 9]

# Sample from each
n_samples = 8
normal_samples = torch.stack([test_dataset[i][0] for i in normal_indices[:n_samples]]).to(device)
anomaly_samples = torch.stack([test_dataset[i][0] for i in anomaly_indices[:n_samples]]).to(device)

# Compute reconstruction errors
vae.eval()
with torch.no_grad():
    recon_normal, _, _ = vae(normal_samples)
    recon_anomaly, _, _ = vae(anomaly_samples)
    
    # MSE per image
    normal_errors = F.mse_loss(recon_normal, normal_samples, reduction='none').view(n_samples, -1).mean(dim=1)
    anomaly_errors = F.mse_loss(recon_anomaly, anomaly_samples, reduction='none').view(n_samples, -1).mean(dim=1)

# Visualize
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Normal images
for i in range(4):
    axes[0, 0].imshow(normal_samples[i].cpu().squeeze() if i == 0 else normal_samples[i].cpu().squeeze(), cmap='gray')
axes[0, 0].set_title('Normal Samples (0-8)', fontsize=12)
axes[0, 0].axis('off')

# Anomaly images
for i in range(4):
    axes[0, 1].imshow(anomaly_samples[i].cpu().squeeze() if i == 0 else anomaly_samples[i].cpu().squeeze(), cmap='gray')
axes[0, 1].set_title('Anomaly Samples (9)', fontsize=12)
axes[0, 1].axis('off')

# Reconstruction errors
axes[1, 0].bar(range(n_samples), normal_errors.cpu().numpy(), color='green', alpha=0.7)
axes[1, 0].set_xlabel('Sample Index')
axes[1, 0].set_ylabel('Reconstruction Error (MSE)')
axes[1, 0].set_title('Normal Samples - Low Error', fontsize=12)
axes[1, 0].grid(alpha=0.3, axis='y')

axes[1, 1].bar(range(n_samples), anomaly_errors.cpu().numpy(), color='red', alpha=0.7)
axes[1, 1].set_xlabel('Sample Index')
axes[1, 1].set_ylabel('Reconstruction Error (MSE)')
axes[1, 1].set_title('Anomaly Samples - Higher Error', fontsize=12)
axes[1, 1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"Average normal error: {normal_errors.mean().item():.6f}")
print(f"Average anomaly error: {anomaly_errors.mean().item():.6f}")
print(f"Ratio: {anomaly_errors.mean().item() / normal_errors.mean().item():.2f}x")
print("\nNote: In real anomaly detection, train only on normal data.")

## Part 15: Experiments and Exploration

### Experiment 1: Effect of Latent Dimension Size

How does bottleneck size affect reconstruction quality?

**Exploration Ideas:**

Try training autoencoders with different latent dimensions:
- `latent_dim = 2`: Extreme compression (392× compression!)
- `latent_dim = 8`: Very small
- `latent_dim = 32`: Our default
- `latent_dim = 128`: Larger capacity

**Questions to explore:**
1. What happens with `latent_dim = 2`? Can you visualize the 2D latent space directly?
2. At what dimension does reconstruction quality plateau?
3. Does the relationship hold for VAE and VQ-VAE?

**Code to try:**
```python
# Train with latent_dim=2 for direct 2D visualization
tiny_ae = VanillaAutoencoder(latent_dim=2).to(device)
# ... train ...
# Extract latents and plot directly (no t-SNE needed!)
```

### Experiment 2: Architecture Depth

**Questions:**
- What if we add more layers to encoder/decoder?
- What if we make the network shallower?
- Does depth matter more for VAE or VQ-VAE?

**Code to try:**
```python
# Deeper encoder
self.encoder = nn.Sequential(
    nn.Linear(784, 512),
    nn.ReLU(),
    nn.Linear(512, 384),
    nn.ReLU(),
    nn.Linear(384, 256),
    nn.ReLU(),
    nn.Linear(256, 128),
    nn.ReLU(),
    nn.Linear(128, latent_dim)
)
```

### Experiment 3: VQ-VAE Codebook Size

**Questions:**
- What happens with very small codebooks (e.g., 64 vectors)?
- What happens with very large codebooks (e.g., 2048 vectors)?
- How does codebook size affect perplexity and reconstruction?

**Code to try:**
```python
small_vqvae = VQVAE(latent_dim=32, num_embeddings=64).to(device)
large_vqvae = VQVAE(latent_dim=32, num_embeddings=2048).to(device)
```

### Experiment 4: Latent Space Arithmetic

Can we perform meaningful operations in latent space?

**Example:** Find the "direction" that transforms one digit to another.

```python
# Encode multiple 3s and 8s
z_3s = [vae.encode(img_of_3) for ...]
z_8s = [vae.encode(img_of_8) for ...]

# Average latent vectors
avg_z_3 = torch.mean(torch.stack(z_3s), dim=0)
avg_z_8 = torch.mean(torch.stack(z_8s), dim=0)

# Compute direction
direction = avg_z_8 - avg_z_3

# Apply to new 3
new_3 = test_dataset[...][0]
z_new_3 = vae.encode(new_3)
z_modified = z_new_3 + 0.5 * direction
modified_img = vae.decode(z_modified)
# Does it look like an 8?
```

## Part 16: Reflection Questions

1. **Compression vs Quality Trade-off:**
   - We compressed 784 dimensions to 32 (24.5× compression). What would happen with 784→2 dimensions? Try it!
   - Is there a "sweet spot" for latent dimension?

2. **Vanilla AE vs VAE:**
   - Why does VAE generate better samples than vanilla AE?
   - What is the role of KL divergence in creating a structured latent space?
   - When would you prefer vanilla AE over VAE?

3. **VQ-VAE Advantages:**
   - Why use discrete latent space instead of continuous?
   - How could VQ-VAE be used for image generation (hint: learn a prior over discrete codes)?
   - What does low codebook perplexity tell us?

4. **Applications:**
   - For denoising, would VQ-VAE work better than VAE? Why or why not?
   - For anomaly detection, should we train on all classes or only "normal" ones?
   - What other applications can you think of?

5. **Latent Space:**
   - Why do similar digits cluster in latent space?
   - What makes interpolation smooth in VAE but not vanilla AE?
   - Can we control specific attributes by manipulating latent dimensions?

## Part 17: Summary and Key Takeaways

### What We Learned

1. **Autoencoders** compress data to a lower-dimensional latent space and reconstruct it
2. **Vanilla AE** learns deterministic encoding but has unstructured latent space
3. **VAE** uses probabilistic encoding with KL regularization for smooth, structured latent space
4. **VQ-VAE** uses discrete latent space (codebook) for better reconstructions and generation
5. **Latent space analysis** reveals semantic structure and enables interpolation
6. **Applications** include compression, denoising, anomaly detection, and generation

### Key Insights

- **Bottleneck size** controls compression-quality trade-off
- **KL divergence** structures VAE latent space for generation
- **Codebook perplexity** indicates healthy VQ-VAE training
- **Reconstruction error** can detect anomalies
- **Interpolation quality** depends on latent space structure

### Next Steps

- **Convolutional autoencoders** for better image processing
- **β-VAE** for disentangled representations
- **VQ-VAE-2** with hierarchical latents
- **Diffusion models** as modern alternative to VAEs
- **Transformers** with VQ-VAE for image generation (DALL-E)

### Further Reading

- Original VAE paper: Kingma & Welling (2013)
- VQ-VAE paper: van den Oord et al. (2017)
- Tutorial on VAEs: Doersch (2016)
- DALL-E: Ramesh et al. (2021)

## Congratulations!

You've completed a comprehensive exploration of image reconstruction with autoencoders. You've:
- ✅ Implemented three types of autoencoders from scratch
- ✅ Analyzed and visualized latent spaces
- ✅ Performed latent space interpolation and manipulation
- ✅ Compared different autoencoder architectures
- ✅ Applied autoencoders to real-world tasks

Keep experimenting and exploring!